# Geolocation Hackathon -- Kaggle Training Notebook

Self-contained: clones the project repo, locates the Kaggle-mounted
competition dataset, then runs geocell setup -> training -> calibration ->
offline inference -> submission CSV.

**Why this exists**: local training (RTX 3050 laptop, 6GB) kept crashing
the machine under sustained load -- repeated full OS reboots mid-run. This
notebook picks up the in-progress resolution/capacity investigation
(224px -> 336px -> 448px -> 518px, each step winning, diminishing returns
past ~448px; more unfrozen DINOv2 blocks + regularization) on Kaggle's
free T4/P100 instead.

**Network use here is a development-time convenience** (`git clone`,
`pip install`), not part of the guess-producing inference path -- the
actual `predict.py` inference call later in this notebook makes zero
network calls, consistent with the project's offline-inference rule (see
README.md guardrails). If you need a network-disabled competition
submission notebook specifically, that's a separate, smaller artifact
(bundle the trained checkpoint + calibration table, skip the clone/install
cells) -- not what this notebook is for.

**Repo**: https://github.com/aryannzzz/geoguessr


## 1. Setup: clone the repo, install dependencies

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/aryannzzz/geoguessr.git"
REPO_DIR = "/kaggle/working/geoguessr"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("cwd:", os.getcwd())
print("HEAD:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                               capture_output=True, text=True).stdout.strip())


In [ ]:
# Install deps, skipping torch/torchvision: Kaggle images ship a
# CUDA-matched torch build already, and overwriting it with our local
# pin risks breaking GPU access. Everything else (timm, shapely,
# geopandas, s2sphere, scipy, albumentations, imagehash, ...) is not
# preinstalled and does need a real install.
#
# requirements.txt has inline comments on several lines (e.g.
# "s2sphere==0.2.5          # S2 geocell indexing") -- strip everything
# from the first "#" onward, not just whole-line comments, or pip
# receives the trailing comment as part of the spec and rejects it.
specs = []
with open("requirements.txt") as f:
    for raw_line in f:
        line = raw_line.split("#", 1)[0].strip()
        if line:
            specs.append(line)

skip_prefixes = ("torch==", "torchvision==")
to_install = [s for s in specs if not s.startswith(skip_prefixes)]
print(f"resolved {len(to_install)} installable specs (skipping torch/torchvision):")
for s in to_install:
    print(" ", s)

req_path = "requirements_kaggle.txt"
with open(req_path, "w") as f:
    f.write("\n".join(to_install) + "\n")

subprocess.run(["pip", "install", "-q", "-r", req_path], check=True)

import torch
print(f"torch {torch.__version__}, cuda available: {torch.cuda.is_available()}, "
      f"device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
assert torch.cuda.is_available(), "No GPU visible -- check Kaggle notebook settings (Settings > Accelerator > GPU T4/P100)"


## 2. Locate the Kaggle-mounted dataset

We don't hardcode a dataset slug/path here -- Kaggle's exact mount
structure for this competition isn't known in advance from this repo
alone (and guessing a wrong path would silently break the notebook). This
searches everything under `/kaggle/input/` for directories that actually
contain our expected image files, matched by filename against the image
IDs we already have on record:

- **training images**: matched against `data/processed/train_manifest_geocells.csv`'s
  `image_id` column (this manifest -- lat/lon, country, geocell
  assignment, tangent-plane offsets -- is already committed to the repo,
  built locally in Phase 1/3, so we only need to find the *image files*
  here, not re-derive labels)
- **test images**: matched against `data/raw/sample_submission.csv`'s
  `image_id` column (also already committed)

Once found, both directories get symlinked into the exact local paths
`src/data/dataset.py` and `src/inference/predict.py` already expect, so
**no source code needs to change** for the path switch -- same code path
as local runs.


In [ ]:
import os
from pathlib import Path
import pandas as pd

ROOT = Path(REPO_DIR)
KAGGLE_INPUT = Path("/kaggle/input")

def find_image_dir(root, expected_filenames, sample_size=30, min_match=5):
    """Walk `root`, return the directory containing the most files
    matching a random sample of `expected_filenames`."""
    import random
    sample = set(random.Random(42).sample(list(expected_filenames), min(sample_size, len(expected_filenames))))
    best_dir, best_count = None, 0
    for dirpath, _, filenames in os.walk(root):
        hit = sample & set(filenames)
        if len(hit) > best_count:
            best_count, best_dir = len(hit), dirpath
    return (best_dir, best_count) if best_count >= min_match else (None, best_count)

train_manifest = pd.read_csv(ROOT / "data" / "processed" / "train_manifest_geocells.csv")
sample_sub = pd.read_csv(ROOT / "data" / "raw" / "sample_submission.csv")

train_img_dir, train_hits = find_image_dir(KAGGLE_INPUT, train_manifest["image_id"].tolist())
test_img_dir, test_hits = find_image_dir(KAGGLE_INPUT, sample_sub["image_id"].tolist())

print(f"train images: {train_img_dir}  ({train_hits} sample filenames matched)")
print(f"test images:  {test_img_dir}  ({test_hits} sample filenames matched)")

assert train_img_dir is not None, (
    "Could not locate training images under /kaggle/input -- check the "
    "dataset is attached to this notebook (Add Input on the right panel), "
    "then re-run this cell. If images live under an unusual nested path, "
    "os.walk should still find them; if not, print `os.listdir('/kaggle/input')` "
    "to inspect the mount and adjust `find_image_dir`'s root."
)
assert test_img_dir is not None, "Could not locate test images under /kaggle/input -- see note above."


In [ ]:
# Symlink into the exact local paths the existing (unmodified) src/
# code expects -- this is the only environment-specific shim needed.
expected_train_dir = ROOT / "data" / "raw" / "training_dataset" / "noised_dataset" / "images"
expected_test_dir = ROOT / "data" / "raw" / "test_images_sampled"

expected_train_dir.parent.mkdir(parents=True, exist_ok=True)
if expected_train_dir.exists() or expected_train_dir.is_symlink():
    expected_train_dir.unlink() if expected_train_dir.is_symlink() else None
if not expected_train_dir.exists():
    expected_train_dir.symlink_to(train_img_dir)

if expected_test_dir.exists() or expected_test_dir.is_symlink():
    expected_test_dir.unlink() if expected_test_dir.is_symlink() else None
if not expected_test_dir.exists():
    expected_test_dir.symlink_to(test_img_dir)

print("train images ->", expected_train_dir, "->", os.path.realpath(expected_train_dir))
print("test images  ->", expected_test_dir, "->", os.path.realpath(expected_test_dir))
print("train file count:", len(os.listdir(expected_train_dir)))
print("test file count:", len(os.listdir(expected_test_dir)))


## 3. Run config -- edit these to change resolution / unfrozen blocks / epochs

Defaults below pick up the in-progress local experiment (448px, 6 of 12
DINOv2 blocks unfrozen, cosine LR schedule, early stopping) rather than
restarting from the 224px/2-block baseline. Kaggle's T4/P100 have ~2.5x
the VRAM of the local 3050 (16GB vs 6GB) that this config was tuned
against (5.7GB peak there) -- there's room to push resolution/batch size
further if this run comes back well short of val Haversine's noise floor;
just re-edit this cell and re-run from here.


In [ ]:
import yaml

RUN_NAME = "kaggle_448_deep"

IMG_SIZE = 448              # 224 -> 336 -> 448 -> 518 all won in matched local
                             # tests; 448 was the cost/benefit pick (518's extra
                             # ~2.4% gain wasn't worth ~25% more time/epoch)
N_UNFROZEN_BLOCKS = 6        # of 12 total DINOv2-small blocks (baseline used 2,
                             # which overfit by epoch 4-5 -- more capacity +
                             # more regularization below is the intended offset)
EPOCHS = 16                  # hard cap -- early stopping (below) should end this
                             # sooner; don't raise this without a reason, time is
                             # the binding constraint this session
BATCH_SIZE = 48
LR_BACKBONE = 1.0e-5
LR_HEADS = 1.0e-3
WEIGHT_DECAY = 0.03          # up from baseline's 0.01 -- more capacity needs more
                             # regularization pressure
LABEL_SMOOTHING = 0.05
LR_SCHEDULE = "cosine"
PATIENCE = 4                 # early-stop if val Haversine median doesn't improve
                             # by > EARLY_STOP_MIN_DELTA_KM for this many epochs
EARLY_STOP_MIN_DELTA_KM = 1.0
NUM_WORKERS = 4               # Kaggle CPU allocation is more limited than a local
                              # 12-core desktop; 4 is a safer default than the
                              # local config's 6 -- raise if you confirm headroom

cfg = {
    "run_name": RUN_NAME,
    "seed": 42,
    "backbone_name": "vit_small_patch14_dinov2.lvd142m",
    "img_size": IMG_SIZE,
    "n_unfrozen_blocks": N_UNFROZEN_BLOCKS,
    "offset_scale_km": 500.0,
    "val_frac": 0.1,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "epochs": EPOCHS,
    "lr_backbone": LR_BACKBONE,
    "lr_heads": LR_HEADS,
    "weight_decay": WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
    "cls_loss_weight": 1.0,
    "reg_loss_weight": 1.0,
    "grad_clip_norm": 1.0,
    "lr_schedule": LR_SCHEDULE,
    "patience": PATIENCE,
    "early_stop_min_delta": EARLY_STOP_MIN_DELTA_KM,
    "amp": True,
}

CONFIG_PATH = ROOT / "configs" / "kaggle_run.yaml"
with open(CONFIG_PATH, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"wrote {CONFIG_PATH}")
print(yaml.safe_dump(cfg, sort_keys=False))


## 4. Geocells

`data/processed/train_manifest_geocells.csv` and `geocell_centroids.csv`
are already committed (built locally in Phase 1/3 -- adaptive k-means,
k=150 merged to 147 cells, min size 15, tangent-plane offset targets) --
default behavior is to just use them, no rebuild. Flip `REBUILD_GEOCELLS`
only if you're deliberately changing the geocell scheme.


In [ ]:
REBUILD_GEOCELLS = False

processed_dir = ROOT / "data" / "processed"
manifest_path = processed_dir / "train_manifest_geocells.csv"
centroids_path = processed_dir / "geocell_centroids.csv"

if REBUILD_GEOCELLS or not (manifest_path.exists() and centroids_path.exists()):
    print("rebuilding geocells from scratch...")
    # build_manifest.build() needs the raw ground-truth coordinates CSV,
    # which isn't committed (only the already-merged processed manifest
    # is). Best-effort: locate it under /kaggle/input by column signature
    # rather than a hardcoded filename.
    gt_dir, gt_hits = None, 0
    for dirpath, _, filenames in os.walk(KAGGLE_INPUT):
        for fn in filenames:
            if fn.endswith(".csv") and "ground_truth" in fn.lower():
                gt_dir = Path(dirpath) / fn
                break
        if gt_dir:
            break
    assert gt_dir is not None, (
        "REBUILD_GEOCELLS=True but couldn't find a ground-truth coordinates "
        "CSV under /kaggle/input -- either locate it manually and adjust "
        "this cell, or set REBUILD_GEOCELLS=False to use the committed "
        "pre-built manifest instead (recommended -- it's already correct)."
    )
    expected_gt_path = ROOT / "data" / "raw" / "training_dataset" / "noised_dataset" / "ground_truth_coordinates.csv"
    expected_gt_path.parent.mkdir(parents=True, exist_ok=True)
    if not expected_gt_path.exists():
        expected_gt_path.symlink_to(gt_dir)

    from src.data.build_manifest import build as build_manifest
    from src.data.geocells import build_geocells
    build_manifest()
    df = pd.read_csv(processed_dir / "train_manifest.csv")
    df, cell_table, n_cells = build_geocells(df)
    df.to_csv(manifest_path, index=False)
    cell_table.to_csv(centroids_path, index=False)
    print(f"rebuilt: {n_cells} geocells")
else:
    print("using committed geocells:", manifest_path, centroids_path)
    print(pd.read_csv(centroids_path).shape[0], "geocells")


## 5. Train

Runs `src/train.py`'s training loop unmodified (same code as local runs)
against the config written above. Checkpoints save every epoch to
`checkpoints/{run_name}_last.pt` and `checkpoints/{run_name}_best.pt`
(best = lowest val Haversine median so far); both are under
`/kaggle/working/geoguessr/checkpoints/`, so they'll show up in this
notebook's **Output** tab automatically once you commit the run.


In [ ]:
from src.train import main as train_main

train_main(str(CONFIG_PATH))


## 6. Calibrate radius (confidence-bucketed quantiles)

In [ ]:
from src.calibrate import main as calibrate_main

calibrate_main(str(CONFIG_PATH))


## 7. Offline inference -> submission CSV

Zero network calls in this step -- loads the checkpoint + calibration
table from local disk and runs inference over the (symlinked) test image
directory, same code path as local runs.


In [ ]:
from src.inference.predict import main as predict_main

out_path = ROOT / "outputs" / "submissions" / f"{RUN_NAME}_submission.csv"
predict_main(str(CONFIG_PATH), out_path=str(out_path))

import pandas as pd
sub = pd.read_csv(out_path)
print(sub.shape)
sub.head()


## 8. Collect the run's key artifacts for download

Everything under `/kaggle/working/geoguessr/` is already in this
notebook's Output tab once you **commit** the notebook (Kaggle persists
the full working directory) -- this cell just also copies the handful of
files you actually need into one flat folder so you don't have to dig
through the repo structure to find them.

**When this finishes: go to the notebook's Output tab (top right, after
committing/running) and download the `final_submission/` folder** -- it
has the submission CSV, the trained checkpoint, the training history, and
the calibration table.


In [ ]:
import shutil

final_dir = Path("/kaggle/working/final_submission")
final_dir.mkdir(exist_ok=True)

artifacts = {
    f"{RUN_NAME}_submission.csv": ROOT / "outputs" / "submissions" / f"{RUN_NAME}_submission.csv",
    f"{RUN_NAME}_best.pt": ROOT / "checkpoints" / f"{RUN_NAME}_best.pt",
    f"{RUN_NAME}_history.csv": ROOT / "outputs" / "training" / f"{RUN_NAME}_history.csv",
    f"{RUN_NAME}_calibration.json": ROOT / "outputs" / "calibration" / f"{RUN_NAME}_calibration.json",
    f"{RUN_NAME}_config_used.json": ROOT / "outputs" / "training" / f"{RUN_NAME}_config_used.json",
}

for dest_name, src_path in artifacts.items():
    if src_path.exists():
        shutil.copy(src_path, final_dir / dest_name)
        print("copied:", dest_name)
    else:
        print("MISSING (check earlier cells for errors):", src_path)

print(f"\nDone. Download everything in the Output tab under final_submission/ "
      f"-- {final_dir}")
